In [5]:
import json
with open('dataforchunk/crud/qwen25_14B_set.json', 'r', encoding='utf-8') as file:  
    qa_data = json.load(file)
len_sents=0
len_lists=0
for sentence in qa_data:
    len_sents+=len(sentence)
    len_lists+=1
len_sents/len_lists,len_sents

(178.35717740162673, 8113468)

In [ ]:
# -*- coding: utf-8 -*-
import evaluate
import os
import json
from tqdm import tqdm
import jieba
def bleu_score(
    continuation: str,
    reference: str,
    with_penalty = False
) -> float:
    f = lambda text: list(jieba.cut(text))
    bleu = evaluate.load('src/.cache/huggingface/bleu')
    results = bleu.compute(predictions=[continuation], references=[[reference]], tokenizer=f)
    
    bleu_avg = results['bleu']
    bleu1 = results['precisions'][0]
    bleu2 = results['precisions'][1]
    bleu3 = results['precisions'][2]
    bleu4 = results['precisions'][3]
    brevity_penalty = results['brevity_penalty']

    if with_penalty:
        return bleu_avg, bleu1, bleu2, bleu3, bleu4
    else:
        return 0.0 if brevity_penalty==0 else bleu_avg/brevity_penalty, bleu1, bleu2, bleu3, bleu4

def rougeL_score(
    continuation: str,
    reference: str
) -> float:
    f = lambda text: list(jieba.cut(text))
    rouge = evaluate.load('src/.cache/huggingface/rouge')
    results = rouge.compute(predictions=[continuation], references=[[reference]], tokenizer=f, rouge_types=['rougeL'])
    score = results['rougeL']
    return score

folder_path = 'dataforchunk/qchunker/eval'
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        print(file_path)
        with open(file_path, 'r', encoding='utf-8') as file:  
            qa_data = json.load(file)
        score_list=[] 
        all_bleu_avg=0
        all_bleu1=0
        all_bleu2=0
        all_bleu3=0
        all_bleu4=0
        num=0
        for qa in tqdm(qa_data):
            query = qa['answer']
            llm_ans = qa['llm_ans']  

            rougeL=rougeL_score(llm_ans,query)
            bleu_avg, bleu1, bleu2, bleu3, bleu4=bleu_score(llm_ans,query,True)
            all_bleu_avg+=bleu_avg
            all_bleu1+=bleu1
            all_bleu2+=bleu2
            all_bleu3+=bleu3
            all_bleu4+=bleu4
            score_list.append(rougeL)
            num+=1
        print(filename,',avg_rougeL: ',sum(score_list)/len(score_list),flush=True)
        print(filename,',avg_bleu_avg: ',all_bleu_avg/num,flush=True)
        print(filename,',avg_bleu1: ',all_bleu1/num,flush=True)
        print(filename,',avg_bleu2: ',all_bleu2/num,flush=True)
        print(filename,',avg_bleu3: ',all_bleu3/num,flush=True)
        print(filename,',avg_bleu4: ',all_bleu4/num,flush=True)


In [ ]:
# -*- coding: utf-8 -*-
from nltk.translate.meteor_score import meteor_score
import jieba
import os
import json
from tqdm import tqdm
import nltk  
nltk.data.path.append('nltk_data')

def preprocess_text(text):
    return ' '.join(jieba.cut(text)).split()

folder_path = 'dataforchunk/qchunker/eval'
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8') as file:  
            qa_data = json.load(file)
        score_list=[] 
        for qa in tqdm(qa_data):
            query = qa['answer']
            llm_ans = qa['llm_ans']  

            processed_reference = preprocess_text(query)
            processed_candidate = preprocess_text(llm_ans)

            score = meteor_score([processed_reference], processed_candidate)
            score_list.append(score)
        print(filename,',avg_sim: ',sum(score_list)/len(score_list),flush=True)

In [ ]:
from nltk.translate.meteor_score import meteor_score
import jieba
import os
import json
from tqdm import tqdm
import nltk  
nltk.data.path.append('nltk_data')
def preprocess_text(text):
    return ' '.join(jieba.cut(text)).split()

folder_path = 'log/tmp1/qchunker_crud_cot_ratio_top8_4_Qwen_7B_Chat'
for root, dirs, files in os.walk(folder_path):  
    for file in files:  
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as file:  
                qa_data = json.load(file)
            score_list=[] 
            for qa in tqdm(qa_data["results"]):
                query = qa["log"]['ground_truth_text']
                llm_ans = qa["log"]['generated_text']  

                processed_reference = preprocess_text(query)
                processed_candidate = preprocess_text(llm_ans)

                score = meteor_score([processed_reference], processed_candidate)
                score_list.append(score)
            print(file_path,',avg_sim: ',sum(score_list)/len(score_list),flush=True)

In [ ]:
import os
import json
from tqdm import tqdm
with open('data/crud_split/split_merged.json', 'r', encoding='utf-8') as file:  
    ques = json.load(file)
ques_dist={}
for item in ques["questanswer_1doc"]:
    ques_dist[item["ID"]]=item["questions"]
folder_path = 'log/tmp2/qchunker_crud_ratio_top8_4_Qwen_7B_Chat'
for root, dirs, files in os.walk(folder_path):  
    for file in files:  
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as file:  
                qa_data = json.load(file)
            score_list=[] 
            for qa in tqdm(qa_data["results"]):
                question=ques_dist[qa["id"]]
                query = qa["log"]['ground_truth_text']
                llm_ans = qa["log"]['generated_text']  
                score_list.append({"question":question,"llm_ans":llm_ans,"answer":query})
            with open('output/'+file_path.split('/')[1]+'.json', 'w', encoding='utf-8') as f:
                json.dump(score_list, f, indent=4, ensure_ascii=False)


In [27]:
import json
with open('dataforchunk/qchunker/aaa/huagong_ratio.json', 'r', encoding='utf-8') as file:  
    qa_data = json.load(file)
all_data={}
for item in qa_data:
    for i,iit in enumerate(item["final_chunk"]):
        all_data[item["outline"][i].replace('\n',' ').strip()+'\n'+iit.replace('\n\n','\n').strip()]=item["final_chunk"]
with open('dataforchunk/qchunker/bbb/huagong_ratio.json', 'w', encoding='utf-8') as f:
    json.dump(all_data, f, indent=4, ensure_ascii=False)